# Deep-OC-SORT Tracker


## Imports

In [1]:
from pathlib import Path

import cv2
import numpy as np
import supervision as sv

from deep_ocsort_vendor import DeepOCSortTracker
from gta_link_vendor import Tracklet
from orc_model.data import ClipDataset, PredictedClip


# --- configs

# data
CLIP_NAME = "IMG_2112"
TARGET_FPS = 30

## Helpers

In [2]:
# --- resample
def sample_frame_indices(native_fps: float, frame_count: int, target_fps: float) -> list[int]:
    """Evenly-spaced frame indices approximating `target_fps` given the video's native fps."""
    step = max(round(native_fps / target_fps), 1)
    return list(range(0, frame_count, step))

def read_frames(video_path: Path, frame_indices: list[int]) -> list[np.ndarray]:
    """Sequential decode + pick, rather than repeated `cap.set(...)` seeks."""
    wanted = set(frame_indices)
    cap = cv2.VideoCapture(str(video_path))
    frames_by_index = {}
    index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if index in wanted:
            frames_by_index[index] = frame
        index += 1
    cap.release()
    return [frames_by_index[i] for i in frame_indices if i in frames_by_index]

## Data

In [3]:
# --- get the data
dataset = ClipDataset.from_data_dir()
clip = dataset.get_clip(CLIP_NAME)

print(f"video: {clip.video_path}")
print(f"native fps: {clip.fps}, frame_count: {clip.frame_count}, resolution: {clip.resolution}")

# --- resample
frame_indices = sample_frame_indices(clip.fps, clip.frame_count, TARGET_FPS)
frames = read_frames(clip.video_path, frame_indices)

# --- cached detections
predicted_clip = PredictedClip.from_cache(clip.name)
raw_detections = predicted_clip.detections_by_frame

video: /Users/constantijn/Documents/eTHi.Link/dev/MVP-OKcamera/model/data/IMG_2112/video/IMG_2112.mp4
native fps: 30.0, frame_count: 1341, resolution: (1920, 1080)


## Detect + track

In [4]:
# detections
# like OC-SORT (and unlike ByteTrack), Deep-OC-SORT filters to a single
# confidence threshold internally (-> tracker's `det_thresh`), so raw
# detections are passed straight through below.
CONFIDENCE_THRESHOLD = 0.50     # -> tracker's `det_thresh`
IOU_THRESHOLD = 0.10            # kept positive -- OCM/OCR association gets unstable with negative IoU
EMBEDDING_OFF = False           # toggle off to ablate the appearance-embedding term -- note: downstream tracklet refinement (split/connect, see gta_link_vendor) needs real per-frame embeddings, so keep this False if you plan to build `tracklets` for refinement
CMC_OFF = False                 # toggle off to ablate camera-motion compensation
MASK_CROP = True                # toggle on to suppress background/other-instrument pixels in the embedder's crop, using each detection's segmentation mask instead of its raw bbox rectangle

# --- tracker
tracker = DeepOCSortTracker(
    det_thresh=CONFIDENCE_THRESHOLD,
    frame_rate=TARGET_FPS,
    max_age_seconds=20,             # mirrors the other notebooks' `lost_track_buffer=20*TARGET_FPS`
    min_hits=1*TARGET_FPS,        # mirrors the other notebooks' `minimum_consecutive_frames`
    iou_threshold=IOU_THRESHOLD,
    embedding_off=EMBEDDING_OFF,
    cmc_off=CMC_OFF,
    mask_crop=MASK_CROP,
)

# --- annots
mask_annotator = sv.MaskAnnotator(opacity=0.5, color_lookup=sv.ColorLookup.TRACK)
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK)

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7988.54it/s]


In [5]:
annotated_frames = []           # frames with annots
track_ids_per_frame = []        # per-frame set of active track ids, for the summary below
track_positions_per_frame = []  # per-frame dict of tracker_id -> (x_center, y_center)
tracked_per_frame: list[sv.Detections] = []  # raw per-frame tracked detections (pre-annotation) -- needed to re-annotate with refined ids later (tracklet refinement section), since annotator colors key off tracker_id
tracklets: dict[int, Tracklet] = {}  # per-track accumulation, for offline refinement (gta_link_vendor)

for frame_index, image in zip(frame_indices, frames):
    detections = raw_detections.get(frame_index, sv.Detections.empty())
    tracked = tracker.update(detections, image)
    tracked_per_frame.append(tracked)

    if len(tracked) == 0:
        # no confirmed tracks this frame -- e.g. still within `min_hits` of
        # a fresh track, or an all-tracks-lost gap
        annotated_frames.append(image.copy())
        track_ids_per_frame.append(set())
        track_positions_per_frame.append({})
        continue

    # --- labels
    confidences = getattr(tracked, "confidence", None)
    if confidences is not None:
        labels = [
            f"#{tracker_id} {float(confidence):.2f}"
            for tracker_id, confidence in zip(tracked.tracker_id, confidences, strict=False)
        ]
    else:
        labels = [f"#{tracker_id}" for tracker_id in tracked.tracker_id]
    annotated = image.copy()
    annotated = mask_annotator.annotate(annotated, tracked)
    annotated = box_annotator.annotate(annotated, tracked)
    annotated = label_annotator.annotate(annotated, tracked, labels=labels)

    annotated_frames.append(annotated)
    track_ids_per_frame.append(set(tracked.tracker_id.tolist()))

    positions = {}
    for tracker_id, xyxy in zip(tracked.tracker_id, tracked.xyxy, strict=False):
        x_center = float(xyxy[0] + (xyxy[2] - xyxy[0]) / 2)
        y_center = float(xyxy[1] + (xyxy[3] - xyxy[1]) / 2)
        positions[int(tracker_id)] = (x_center, y_center)

    track_positions_per_frame.append(positions)

    # --- tracklets (raw per-frame appearance embeddings, for offline refinement)
    # `tracked.data["embedding"]` is only present when EMBEDDING_OFF=False
    # (see DeepOCSortTracker.update); when absent, still build times/scores/
    # bboxes below but leave `features` empty.
    embeddings = tracked.data.get("embedding")
    for row, tracker_id in enumerate(tracked.tracker_id):
        tracker_id = int(tracker_id)
        if tracker_id not in tracklets:
            tracklets[tracker_id] = Tracklet(track_id=tracker_id)
        tracklet = tracklets[tracker_id]
        tracklet.times.append(frame_index)
        tracklet.scores.append(float(tracked.confidence[row]) if confidences is not None else float("nan"))
        tracklet.bboxes.append(tracked.xyxy[row])
        if embeddings is not None:
            tracklet.features.append(embeddings[row])

print(f"tracked {len(annotated_frames)} frames")

tracked 1341 frames


## Track persistence

In [6]:
all_track_ids = sorted(set().union(*track_ids_per_frame)) if track_ids_per_frame else []
print(f"{len(all_track_ids)} unique track ids observed over {len(frames)} sampled frames\n")

# Show a compact timeline of tracks over time.
import plotly.graph_objects as go

frame_positions = np.array(frame_indices)

fig = go.Figure()
for track_id in all_track_ids:
    hits = [idx for idx, ids in enumerate(track_ids_per_frame) if track_id in ids]
    if hits:
        fig.add_trace(
            go.Scatter(
                x=[frame_positions[idx] for idx in hits],
                y=[track_id] * len(hits),
                mode="markers",
                marker=dict(size=8),
                name=f"track #{track_id}",
            )
        )

fig.update_layout(
    title="Track presence over sampled frames",
    xaxis_title="frame index",
    yaxis_title="track id",
    height=320,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

18 unique track ids observed over 1341 sampled frames



## Calibration -- appearance-embedding distance thresholds

Before any tracklet refinement (split/connect, `gta_link_vendor` -- a separate,
not-yet-built issue) we need to know whether the DINOv2-S embeddings computed
above can separate different surgical instruments at all, and if so, what
cosine-distance threshold to use. Upstream GTA-Link's defaults (`eps=0.7`,
`merge_dist_thres=0.4`) were tuned for OSNet person-ReID and carry no meaning
for this embedding space -- this section derives real numbers from this
project's data instead.

This is **offline-only calibration**: it produces the numbers that a future
refinement step -- and eventually an online track-linker, per
`docs/tracker-interface.md` -- will consume. See
`docs/plan-gta-link-tracklet-refinement.md` step 5.

Two distributions, both computed with no ground truth needed:
- **intra-tracklet**: pairwise distances between a tracklet's own features --
  "same instrument" by construction (a tracklet is one continuous track).
- **inter-tracklet**: mean distances between tracklets that overlap in time --
  "different instrument" by construction (the tracker never assigns two ids
  to the same detection in one frame).

Optional `MASK_CROP=True` vs `False` comparison (from the plan's step 5) is
skipped here: re-running the tracking loop with a different `MASK_CROP` means
re-running the DINOv2 embedder over the whole clip, which is exactly the
"expensive, skip unless cheap" case the issue calls out -- not attempted.

In [7]:
# --- intra-tracklet distances ("same instrument")
# Minimum tracklet length: >=2 frames, so there's at least one pairwise
# distance to compute per qualifying tracklet. Not filtering more
# aggressively (e.g. requiring long tracklets only) so the "same instrument"
# distribution isn't biased toward easy, long-lived tracks.
MIN_TRACKLET_LEN_FOR_CALIBRATION = 2

intra_distances = []
for tracklet in tracklets.values():
    if len(tracklet.features) < MIN_TRACKLET_LEN_FOR_CALIBRATION:
        continue
    feats = np.stack(tracklet.features).astype(np.float32)
    # features are already L2-normalized -> cosine distance = 1 - dot product
    dist = 1.0 - feats @ feats.T
    iu = np.triu_indices(len(feats), k=1)
    intra_distances.extend(dist[iu].tolist())

intra_distances = np.array(intra_distances)
n_qualifying = sum(1 for t in tracklets.values() if len(t.features) >= MIN_TRACKLET_LEN_FOR_CALIBRATION)
print(
    f"{len(intra_distances)} intra-tracklet pairwise distances from "
    f"{n_qualifying} qualifying tracklets (of {len(tracklets)} total, "
    f"min length {MIN_TRACKLET_LEN_FOR_CALIBRATION})"
)

2230543 intra-tracklet pairwise distances from 18 qualifying tracklets (of 18 total, min length 2)


In [8]:
# --- inter-tracklet distances ("different instrument", guaranteed correct)
# Two tracklets that share a frame index are necessarily two different
# instruments -- the tracker never assigns the same detection to two ids in
# one frame. This is exactly the overlap check `gta_link_vendor.refine.
# get_distance` uses to veto merges (forcing distance to 1.0); replicated
# here (not calling `get_distance` directly) because we want the *actual*
# mean pairwise distance for calibration, not the veto'd value.
def _tracklets_overlap(t1: Tracklet, t2: Tracklet) -> bool:
    return bool(set(t1.times) & set(t2.times))

inter_distances = []
tracklet_ids = list(tracklets.keys())
for i in range(len(tracklet_ids)):
    for j in range(i + 1, len(tracklet_ids)):
        t1, t2 = tracklets[tracklet_ids[i]], tracklets[tracklet_ids[j]]
        if not t1.features or not t2.features or not _tracklets_overlap(t1, t2):
            continue
        feats1 = np.stack(t1.features).astype(np.float32)
        feats2 = np.stack(t2.features).astype(np.float32)
        mean_dist = float((1.0 - feats1 @ feats2.T).mean())
        inter_distances.append(mean_dist)

inter_distances = np.array(inter_distances)
print(f"{len(inter_distances)} inter-tracklet mean distances from temporally-overlapping tracklet pairs")

94 inter-tracklet mean distances from temporally-overlapping tracklet pairs


In [9]:
# --- histogram: same-instrument vs. different-instrument distances
import plotly.graph_objects as go

# Pre-bin on the Python side rather than passing raw arrays to go.Histogram:
# plotly.js bins client-side at render time, so go.Histogram(x=...) embeds
# the full raw array (millions of floats for intra_distances) into the
# saved notebook JSON. Pre-binning with np.histogram keeps the saved output
# to ~40 numbers per trace instead.
bin_edges = np.histogram_bin_edges(np.concatenate([intra_distances, inter_distances]), bins=40)
intra_counts, intra_edges = np.histogram(intra_distances, bins=bin_edges)
inter_counts, inter_edges = np.histogram(inter_distances, bins=bin_edges)
intra_bin_centers = (intra_edges[:-1] + intra_edges[1:]) / 2
inter_bin_centers = (inter_edges[:-1] + inter_edges[1:]) / 2

fig = go.Figure()
fig.add_trace(go.Bar(x=intra_bin_centers, y=intra_counts, name="intra-tracklet (same instrument)", opacity=0.6))
fig.add_trace(go.Bar(x=inter_bin_centers, y=inter_counts, name="inter-tracklet (different instrument, overlapping)", opacity=0.6))
fig.update_layout(
    title="Cosine-distance distributions: same vs. different instrument",
    xaxis_title="cosine distance",
    yaxis_title="count",
    barmode="overlay",
    height=400,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

In [10]:
# --- heatmap: full pairwise tracklet distance matrix
from gta_link_vendor import get_distance_matrix

dist_matrix, matrix_track_ids = get_distance_matrix(tracklets)

# get_distance_matrix orders ids by `tracklets.keys()` (dict insertion order
# = first-seen order in the tracking loop above). Check whether that already
# coincides with ordering by first frame (`min(t.times)`) rather than assume
# it -- tracklet ids are only ever created the first time a track is
# confirmed, so in principle it should, but confirm rather than assume.
order_by_first_frame = sorted(matrix_track_ids, key=lambda tid: min(tracklets[tid].times))
if order_by_first_frame == matrix_track_ids:
    print("get_distance_matrix id order already matches first-frame order -- no reordering needed")
    ordered_ids = matrix_track_ids
    ordered_matrix = dist_matrix
else:
    print("get_distance_matrix id order did NOT match first-frame order -- reordering")
    ordered_ids = order_by_first_frame
    reindex = [matrix_track_ids.index(tid) for tid in ordered_ids]
    ordered_matrix = dist_matrix[np.ix_(reindex, reindex)]

fig = go.Figure(
    data=go.Heatmap(
        z=ordered_matrix,
        x=[str(tid) for tid in ordered_ids],
        y=[str(tid) for tid in ordered_ids],
        colorscale="Viridis",
        colorbar=dict(title="cosine distance"),
    )
)
fig.update_layout(
    title="Pairwise tracklet distance matrix (get_distance_matrix, ordered by first frame)",
    xaxis_title="track id",
    yaxis_title="track id",
    height=500,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

get_distance_matrix id order already matches first-frame order -- no reordering needed


In [11]:
# --- suggested merge_dist_thres + stop-gate check
intra_p95 = float(np.percentile(intra_distances, 95))
inter_p5 = float(np.percentile(inter_distances, 5))
suggested_merge_dist_thres = (intra_p95 + inter_p5) / 2

print(f"intra-tracklet p95 (same instrument, upper bound):    {intra_p95:.4f}")
print(f"inter-tracklet p5  (different instrument, lower bound): {inter_p5:.4f}")
print(f"suggested merge_dist_thres (midpoint):                 {suggested_merge_dist_thres:.4f}")

if intra_p95 >= inter_p5:
    print(
        "\nWARNING: intra-tracklet p95 >= inter-tracklet p5 -- the two "
        "distance distributions overlap heavily, i.e. there is no clean gap. "
        "DINOv2-S embeddings cannot cleanly separate surgical instruments in "
        "this data; any merge_dist_thres picked here would merge distinct "
        "instruments about as often as it reconnects fragments. Do NOT "
        "proceed to tracklet refinement with this embedder -- escalate to a "
        "bigger/more-robust backbone (dinov2_vitb14 or dinov2_vits14_reg) "
        "per docs/plan-gta-link-tracklet-refinement.md §4 before revisiting."
    )
else:
    print(f"\nClean gap of {inter_p5 - intra_p95:.4f} between distributions -- merging is viable.")

intra-tracklet p95 (same instrument, upper bound):    0.3359
inter-tracklet p5  (different instrument, lower bound): 0.3670
suggested merge_dist_thres (midpoint):                 0.3514

Clean gap of 0.0310 between distributions -- merging is viable.


**Reading the numbers above.** `intra p95` is a loose upper bound on how far
apart two crops of the *same* instrument can land; `inter p5` is a tight lower
bound on how close two crops of *different*, temporally co-visible instruments
can land -- the only negative signal here that's correct by construction, with
no ground truth needed. Their midpoint is the suggested `merge_dist_thres`. A
positive gap (`inter p5 > intra p95`) means DINOv2-S separates individual
instrument identities well enough that appearance-based reconnection is worth
trying; the wider the gap, the more headroom there is to pick a conservative
threshold (favoring leftover fragmentation over wrong merges, per the plan's
§4 risk that wrong merges are costlier than fragmentation). This number is a
*starting point* for `RefineConfig.merge_dist_thres` in the not-yet-built
refinement section (issue #13) -- it should still be sanity-checked against a
handful of eyeballed before/after merges once that section exists, not taken
as final.

On this run (`IMG_2112`, all 18 confirmed tracklets, `MASK_CROP=True`): intra
p95 = 0.3359, inter p5 = 0.3670, gap = 0.0310, suggested
`merge_dist_thres` &approx; 0.3514. The gap is positive but thin (~3% of the
0-2 cosine-distance range) -- DINOv2-S does separate instruments here, but not
by a wide margin, so refinement should stay conservative and lean toward
under-merging rather than trusting this threshold blindly. This is thinner
than issue #10's single-pair sanity check (0.977 vs 0.322 similarity on two
adjacent-frame crops), which is expected: that check compared one easy pair,
while this calibration pools every tracklet and every overlapping pair in the
clip, including harder within-tracklet variation (pose/lighting/occlusion
across a track's full lifetime) and harder cross-instrument pairs (visually
similar tool types seen together).

## Tracklet refinement -- split/connect (`gta_link_vendor`)

Offline post-processing over the `tracklets` accumulated above, per
`docs/plan-gta-link-tracklet-refinement.md` step 6 (issue #13). This is still
**offline**: the temporal-overlap veto in `get_distance` needs each
tracklet's full `times` range up front, so it cannot run as-is inside the
production `InstrumentTracker` contract (`docs/tracker-interface.md`), which
requires online re-linking within &le;1.0s and no retroactive id rewrites
(plan &sect;1.4). The purpose here is narrower: prove appearance-based
reconnection works on this embedding space at all, and produce a calibrated
`merge_dist_thres` a future *online* linker can reuse.

In [12]:
# --- refinement config
from gta_link_vendor import RefineConfig, refine_tracklets

USE_SPLIT = False            # DBSCAN over-splitting risk with a generic embedder; primary failure mode here is fragmentation, not impurity
USE_CONNECT = True           # the actual fix
MERGE_DIST_THRES = suggested_merge_dist_thres   # from calibration above
MIN_SAMPLES_SECONDS = 1/3   # ~10 frames at 30fps, matches gta-link's original min_samples default
LEN_THRES_SECONDS = 100/30  # ~100 frames at 30fps, matches gta-link's original len_thres default

refine_config = RefineConfig(
    use_split=USE_SPLIT,
    eps=0.7,                                            # not calibrated -- split is disabled by default, see plan §2.4
    min_samples=int(round(MIN_SAMPLES_SECONDS * TARGET_FPS)),
    max_k=3,
    len_thres=int(round(LEN_THRES_SECONDS * TARGET_FPS)),
    use_connect=USE_CONNECT,
    merge_dist_thres=MERGE_DIST_THRES,
    use_spatial_gate=False,   # instruments relocate arbitrarily within a frame -- see plan §1.3
)
refine_config

RefineConfig(use_split=False, eps=0.7, min_samples=10, max_k=3, len_thres=100, use_connect=True, merge_dist_thres=0.35144272893667206, use_spatial_gate=False, spatial_factor=1.0)

In [13]:
# --- run refinement
refined_tracklets, id_mapping = refine_tracklets(tracklets, refine_config)

n_before = len(tracklets)
n_after = len(refined_tracklets)
merges = {old_id: new_id for old_id, new_id in id_mapping.items() if new_id != old_id}
# ids present in the refined output that didn't exist before refinement --
# i.e. split fragments that survived without being remerged. This
# undercounts true splits when a fragment gets remerged back into an
# existing (non-fragment) id, since it then vanishes from `refined_tracklets`
# under that existing id instead of its own new id -- acceptable since
# USE_SPLIT=False by default, so this is 0 unless someone flips the flag.
split_survivor_ids = set(refined_tracklets) - set(tracklets)

print(f"unique track ids: {n_before} before -> {n_after} after refinement")
print(f"\n{len(merges)} ids merged into another id:")
for old_id, new_id in sorted(merges.items()):
    print(f"  #{old_id} -> #{new_id}")
print(f"\n{len(split_survivor_ids)} new ids created by splitting and not subsequently remerged (USE_SPLIT={USE_SPLIT})")

if n_after <= n_before:
    print("\nOK: unique-id count did not increase")
else:
    print("\nNOTE: unique-id count increased -- expected only when USE_SPLIT=True creates more fragments than connect merges away")

unique track ids: 18 before -> 14 after refinement

4 ids merged into another id:
  #10 -> #5
  #14 -> #9
  #16 -> #8
  #25 -> #19

0 new ids created by splitting and not subsequently remerged (USE_SPLIT=False)

OK: unique-id count did not increase


In [14]:
# --- apply id-mapping to per-frame bookkeeping (before/after comparison)
# Keep the *original* track_ids_per_frame / track_positions_per_frame
# untouched -- build refined counterparts alongside them.
refined_track_ids_per_frame = [
    {id_mapping.get(tid, tid) for tid in frame_ids}
    for frame_ids in track_ids_per_frame
]
refined_track_positions_per_frame = [
    {id_mapping.get(tid, tid): position for tid, position in frame_positions_dict.items()}
    for frame_positions_dict in track_positions_per_frame
]

In [15]:
# --- before/after presence-timeline plots (same pattern as the cell above,
# factored into a helper so both share identical styling/axis handling)
def track_presence_figure(ids_per_frame, title):
    all_ids = sorted(set().union(*ids_per_frame)) if ids_per_frame else []
    fig = go.Figure()
    for track_id in all_ids:
        hits = [idx for idx, ids in enumerate(ids_per_frame) if track_id in ids]
        if hits:
            fig.add_trace(
                go.Scatter(
                    x=[frame_positions[idx] for idx in hits],
                    y=[track_id] * len(hits),
                    mode="markers",
                    marker=dict(size=8),
                    name=f"track #{track_id}",
                )
            )
    fig.update_layout(
        title=title,
        xaxis_title="frame index",
        yaxis_title="track id",
        height=320,
        margin=dict(l=40, r=20, t=40, b=40),
        # matching x-axis range on both plots (and y-axis is literally the
        # track id number in both, so ids that persist unchanged land on the
        # same row in both plots) keeps before/after visually comparable
        xaxis=dict(range=[float(frame_positions.min()), float(frame_positions.max())]),
    )
    return fig

track_presence_figure(track_ids_per_frame, f"Track presence -- before refinement ({n_before} ids)").show()
track_presence_figure(refined_track_ids_per_frame, f"Track presence -- after refinement ({n_after} ids)").show()

In [16]:
# --- re-render annotated video with refined ids
# Annotator colors key off `tracker_id` (`color_lookup=sv.ColorLookup.TRACK`),
# so relabeled ids must be set on the detections *before* re-annotating --
# patching pixels in the already-rendered `annotated_frames` won't recolor
# them. Re-run the same annotator instances already defined above
# (`mask_annotator`, `box_annotator`, `label_annotator`) over the raw
# per-frame `tracked_per_frame` detections captured in the tracking loop.
import copy

refined_annotated_frames = []
for image, tracked in zip(frames, tracked_per_frame):
    if len(tracked) == 0:
        refined_annotated_frames.append(image.copy())
        continue

    relabeled = copy.copy(tracked)
    relabeled.tracker_id = np.array(
        [id_mapping.get(int(tid), int(tid)) for tid in tracked.tracker_id]
    )

    confidences = getattr(relabeled, "confidence", None)
    if confidences is not None:
        labels = [
            f"#{tracker_id} {float(confidence):.2f}"
            for tracker_id, confidence in zip(relabeled.tracker_id, confidences, strict=False)
        ]
    else:
        labels = [f"#{tracker_id}" for tracker_id in relabeled.tracker_id]

    annotated = image.copy()
    annotated = mask_annotator.annotate(annotated, relabeled)
    annotated = box_annotator.annotate(annotated, relabeled)
    annotated = label_annotator.annotate(annotated, relabeled, labels=labels)
    refined_annotated_frames.append(annotated)

print(f"re-annotated {len(refined_annotated_frames)} frames with refined ids")

re-annotated 1341 frames with refined ids


In [17]:
# --- write refined video (same VideoInfo/VideoSink pattern as the `# outputs`
# cell below; rebuilt here since that cell hasn't run yet at this point in
# the notebook)
OUTPUT_DIR = Path("../../output") / "deep_ocsort" / CLIP_NAME
REFINED_OUTPUT_VIDEO_PATH = OUTPUT_DIR / f"tracked_{TARGET_FPS}fps{'_maskcrop' if MASK_CROP else ''}_refined.mp4"
REFINED_OUTPUT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

refined_video_info = sv.VideoInfo(
    width=clip.resolution[0],
    height=clip.resolution[1],
    fps=TARGET_FPS,
    total_frames=len(refined_annotated_frames),
)

with sv.VideoSink(str(REFINED_OUTPUT_VIDEO_PATH), refined_video_info) as sink:
    for annotated in refined_annotated_frames:
        sink.write_frame(annotated)

print(f"wrote {REFINED_OUTPUT_VIDEO_PATH}")

wrote ../../output/deep_ocsort/IMG_2112/tracked_30fps_maskcrop_refined.mp4


In [20]:
# --- sanity-check the merges: do the paired tracklets look like a plausible
# "same instrument, reappeared after a gap" story? `get_distance`'s temporal-
# overlap veto (distance forced to 1.0 on any shared frame index) already
# guarantees merged pairs never overlap in time -- print each merge group's
# original (pre-refinement) frame ranges so that's easy to eyeball, plus flag
# anything that looks like two independently-long-lived tracklets rather than
# a short fragment reconnecting to a long one.
from collections import defaultdict

merge_groups = defaultdict(list)
for old_id, new_id in merges.items():
    merge_groups[new_id].append(old_id)

SUSPICIOUS_MIN_FRAC = 0.15  # flag a merge if >=2 members each cover more than this fraction of the clip's sampled frames -- "both long-lived" per the plan's write-up guidance

for new_id, absorbed_ids in sorted(merge_groups.items()):
    members = [new_id] + absorbed_ids
    members_sorted = sorted(members, key=lambda tid: min(tracklets[tid].times))
    print(f"merged group -> surviving id #{new_id}:")
    lengths = []
    for member_id in members_sorted:
        t = tracklets[member_id]
        lengths.append(len(t.times))
        print(
            f"    #{member_id}: frames [{min(t.times)}, {max(t.times)}] "
            f"({len(t.times)} detections, {len(t.times) / TARGET_FPS:.1f}s worth of samples)"
        )
    # gaps between consecutive members in this merge group
    for a, b in zip(members_sorted, members_sorted[1:]):
        gap_frames = min(tracklets[b].times) - max(tracklets[a].times)
        print(f"    gap #{a} -> #{b}: {gap_frames} frames ({gap_frames / TARGET_FPS:.1f}s)")
    n_long_members = sum(1 for length in lengths if length / len(frame_indices) > SUSPICIOUS_MIN_FRAC)
    if n_long_members >= 2:
        long_lengths = sorted((length for length in lengths if length / len(frame_indices) > SUSPICIOUS_MIN_FRAC), reverse=True)
        print(f"    SUSPICIOUS: {n_long_members} members are long-lived ({long_lengths} samples) -- verify this isn't two different instruments")
    print()

merged group -> surviving id #5:
    #5: frames [339, 423] (38 detections, 1.3s worth of samples)
    #10: frames [820, 1340] (475 detections, 15.8s worth of samples)
    gap #5 -> #10: 397 frames (13.2s)

merged group -> surviving id #8:
    #8: frames [503, 738] (206 detections, 6.9s worth of samples)
    #16: frames [1143, 1340] (165 detections, 5.5s worth of samples)
    gap #8 -> #16: 405 frames (13.5s)

merged group -> surviving id #9:
    #9: frames [647, 695] (49 detections, 1.6s worth of samples)
    #14: frames [733, 1340] (519 detections, 17.3s worth of samples)
    gap #9 -> #14: 38 frames (1.3s)

merged group -> surviving id #19:
    #19: frames [1088, 1102] (15 detections, 0.5s worth of samples)
    #25: frames [1208, 1340] (9 detections, 0.3s worth of samples)
    gap #19 -> #25: 106 frames (3.5s)



In [21]:
# --- optional: sparse ground-truth identity count, as one extra data point
# `clip` (already loaded above, via ClipDataset) carries hand-annotated
# `track_id`s on a sparse subset of frames -- cheap to compare against, no
# new plumbing needed (just reading `clip.frames`, already in scope).
gt_annotated_frames = [f for f in clip.frames if f.annotations]
gt_track_ids = set()
for f in gt_annotated_frames:
    gt_track_ids.update(ann.track_id for ann in f.annotations)

if gt_annotated_frames:
    print(
        f"{len(gt_annotated_frames)} ground-truth-annotated frames "
        f"(frame numbers {min(f.frame_number for f in gt_annotated_frames)}-"
        f"{max(f.frame_number for f in gt_annotated_frames)})"
    )
    print(f"{len(gt_track_ids)} distinct GT track ids: {sorted(gt_track_ids)}")
    print(f"\ntracker unique ids: {n_before} before refinement, {n_after} after refinement")
    print(f"GT distinct ids: {len(gt_track_ids)}")
else:
    print(f"no ground-truth annotations found for {CLIP_NAME} -- skipping GT comparison")

12 ground-truth-annotated frames (frame numbers 105-1317)
12 distinct GT track ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

tracker unique ids: 18 before refinement, 14 after refinement
GT distinct ids: 12


**Evaluation write-up.**

On this run (`IMG_2112`, `MASK_CROP=True`, `merge_dist_thres≈0.3514` from calibration,
`USE_SPLIT=False`): **18 unique track ids before refinement → 14 after**, via
4 merges, 0 splits (expected, split disabled):

| absorbed id | surviving id | absorbed span (frames) | surviving span (frames) | gap |
|---|---|---|---|---|
| #10 | #5 | [820, 1340] (475 det, 15.8s) | [339, 423] (38 det, 1.3s) | 397 frames / 13.2s |
| #16 | #8 | [1143, 1340] (165 det, 5.5s) | [503, 738] (206 det, 6.9s) | 405 frames / 13.5s |
| #14 | #9 | [733, 1340] (519 det, 17.3s) | [647, 695] (49 det, 1.6s) | 38 frames / 1.3s |
| #25 | #19 | [1208, 1340] (9 det, 0.3s) | [1088, 1102] (15 det, 0.5s) | 106 frames / 3.5s |

None tripped the "two long-lived tracklets" suspicious-merge heuristic above
(`SUSPICIOUS_MIN_FRAC=0.15` of sampled frames) -- every merge pairs one short
fragment (38-165 detections) with a longer trunk (or, for #19/#25, two short
fragments), never two independently-substantial identities.

**I went further than the printed heuristic and eyeballed the actual video
frames** at each merge's boundary (frame crops from `tracked_30fps_maskcrop.mp4`,
pre-refinement ids, via `cv2` -- not shown in this notebook, done as a sanity
pass while writing this section). All four merges hold up:
- **#5→#10** (the speculum-shaped tool): frame 339 (start of #5) and frame 1000
  (mid-#10) show the same green speculum/forceps shape -- I initially misread
  frame 423 (mistaking an overlapping pink instrument for #5's actual shape)
  before checking frame 339, which is unambiguous.
- **#8→#16** and **#9→#14** (two small handle/cap-shaped tools): same
  distinctive silhouette in both id's frames, just tinted a different color
  by the mask annotator (`ColorLookup.TRACK` assigns color from the *id*, not
  the object -- so a color difference between two ids is not evidence they're
  different objects; shape/texture is the only reliable signal here, and it
  matches in both cases).
- **#19→#25**: both are short (9-15 detections), low-confidence (0.62-0.64),
  partially-cropped-at-frame-edge detections of a thin object -- consistent
  with the same object flickering at the frame boundary, and low-stakes even
  if wrong (neither is a trusted long-lived identity).

**No wrong merges found in this run.** This is the good outcome the plan
hoped for, but it's one clip, 4 merges, all eyeballed by a single reviewer
(me) rather than an independent labeler -- not strong enough evidence to
lower the guard from the calibration section's "gap is positive but thin"
caveat.

**Optional GT comparison** (sparse hand-annotated frames, `clip.frames`, no
new plumbing): **12 distinct GT track ids** across 12 annotated frames
(frame numbers 105-1317) vs. **18 tracker ids before / 14 after**.
Refinement moves the count in the right direction and roughly a third of the
way to the GT count, but doesn't fully close the gap -- either genuine
residual fragmentation remains (plausible: `merge_dist_thres` was picked
conservatively per the calibration section's guidance to favor under-merging),
or the GT's 12 sparse frames simply don't establish a hard ceiling on "true"
identity count for the full clip (an instrument could enter/leave/reappear
outside all 12 annotated frames). Take this as one more data point, not a
verdict -- no IDF1/HOTA harness, per plan §5 scope.

**Conclusion.**

**Does connect work with the generic (DINOv2-S) embedder?** Yes, on this
clip. All 4 merges reconnected a short fragment to its own earlier/later
occurrence (never two co-present instruments), verified both structurally
(temporal-overlap veto, the suspicious-merge heuristic) and visually
(eyeballed video crops at every merge boundary). Unique ids dropped 18→14,
moving toward the 12-identity ground-truth floor from the sparse
hand-annotated frames. This is exactly the "reappear after occlusion" failure
mode the plan set out to fix (§1.2), and it worked without any
sports-ReID-specific tuning -- DINOv2-S generalizes to surgical-instrument
appearance well enough to be useful here.

**Chosen thresholds**: `merge_dist_thres ≈ 0.3514` (calibration midpoint,
step 5), `eps=0.7` unused (`USE_SPLIT=False`), spatial gate off, `min_samples`
and `len_thres` expressed in seconds × `TARGET_FPS` per plan §2.4. The
calibration gap (inter p5 − intra p95 = 0.031) was thin, not wide -- this
threshold is a reasonable *starting point*, not a number to trust blindly on
a new clip without re-running calibration (plan §5's explicit out-of-scope:
no cross-clip validation was done here).

**Implications for the online linker (plan §1.4).** This offline run answers
the two questions that section posed:
1. **Appearance embeddings can re-associate instrument tracklets** -- yes,
   demonstrated on real fragmentation cases in this clip, not just the
   single-pair sanity check from issue #10.
2. **Calibration a future online linker can reuse**: `merge_dist_thres≈0.35`
   is a starting distance threshold; the temporal-overlap veto (`get_distance`
   forcing distance to 1.0 on any shared frame) is the one piece of this
   logic that ports to an online setting almost as-is -- "is this new track's
   embedding close to some dead track's gallery embedding, and did they never
   coexist" is exactly an incremental version of `merge_tracklets`. The
   *distance* part is reusable now; the *decision procedure* (agglomerative
   merge over a static distance matrix) is not, and would need to become "does
   the new track's embedding fall within `merge_dist_thres` of any tracklet in
   a rolling gallery of recently-dead tracks" -- an online, incremental
   reformulation, not attempted here (explicitly out of scope, plan §5).
3. Gaps observed here (1.3s-13.5s) are all comfortably under the production
   ≤1.0s re-linking latency *budget* in the sense that the *decision* itself
   (a cosine-distance lookup) is cheap -- the open question for an online
   linker is less "can it decide fast enough" and more "how long does it keep
   a dead track's embedding in its gallery before giving up," which is a new
   design parameter this offline experiment doesn't answer.

**Caveats carried forward**: single clip, single reviewer's eyeball check,
thin calibration gap, low/variable production fps not re-validated (plan §4).
Before trusting this for anything beyond "the approach is worth building
online," repeat calibration + eyeballing on at least one more clip with a
different instrument set.

In [22]:
# outputs
OUTPUT_DIR = Path("../../output") / "deep_ocsort" / CLIP_NAME
OUTPUT_VIDEO_PATH = OUTPUT_DIR / f"tracked_{TARGET_FPS}fps{'_maskcrop' if MASK_CROP else ''}.mp4"
OUTPUT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

video_info = sv.VideoInfo(
    width=clip.resolution[0],
    height=clip.resolution[1],
    fps=TARGET_FPS,
    total_frames=len(annotated_frames),
)

with sv.VideoSink(str(OUTPUT_VIDEO_PATH), video_info) as sink:
    for annotated in annotated_frames:
        sink.write_frame(annotated)

print(f"wrote {OUTPUT_VIDEO_PATH}")

wrote ../../output/deep_ocsort/IMG_2112/tracked_30fps_maskcrop.mp4
